In [ ]:
import os
import sys
import glob
import numpy as np
import h5py
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, TensorDataset
from torchvision import transforms
from omegaconf import OmegaConf
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import umap
import matplotlib.pyplot as plt

PROJECT_ROOT = "/u/yacheng/projects/ssl_outthere"
sys.path.insert(0, PROJECT_ROOT)
sys.path.insert(0, os.path.join(PROJECT_ROOT, "encoder_image/astrodino/benchmark/linearprobe"))

from dinov2.eval.setup import build_model_for_eval
from encoder_image.astrodino.train.data.augmentations import ToRGB

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# Pixel scale conversion: catalog radius_sersic is in degrees, image pixel scale is 30 mas
PIXEL_SCALE_MAS = 30  # mas per pixel
DEG_TO_PIXEL = 3600 * 1000 / PIXEL_SCALE_MAS  # = 120000 pixels per degree

## 0. Configuration

In [ ]:
# Model config
MODEL_CONFIG = "/u/yacheng/projects/ssl_outthere/encoder_image/astrodino/model/astrodino_f150w_vitb/config.yaml"
MODEL_WEIGHTS = "/u/yacheng/projects/ssl_outthere/encoder_image/astrodino/model/astrodino_f150w_vitb/eval/training_149999/teacher_checkpoint.pth"
DATA_ROOT = "/u/yacheng/projects/ssl_outthere/images/jwst/f150w"

BATCH_SIZE = 64
MAX_SAMPLES = 20000  # Set to -1 for all samples
SEED = 42

# Error-based filtering for radius_sersic
# Use relative error (err/val) threshold; set to None to disable
RADIUS_SERSIC_REL_ERR_THRESH = 0.01# Keep samples with relative error < 30%

## 1. Check Effective Radius Distribution

In [ ]:
# Load radius_sersic from all h5 files with error-based filtering
h5_files = sorted(glob.glob(os.path.join(DATA_ROOT, "*.h5")))
print(f"Found {len(h5_files)} h5 files")

all_radii = []
all_radii_unfiltered = []

for fpath in tqdm(h5_files, desc="Loading radius_sersic"):
    with h5py.File(fpath, 'r') as f:
        if 'radius_sersic' not in f:
            continue
        rs = f['radius_sersic'][:]
        rs_pix = rs * DEG_TO_PIXEL  # Convert degrees -> pixels

        # Unfiltered (just finite & positive)
        base_mask = np.isfinite(rs) & (rs > 0)
        all_radii_unfiltered.append(rs_pix[base_mask])

        # Error-based filtering
        rs_err = None
        if 'radius_sersic_err' in f:
            rs_err = f['radius_sersic_err'][:]

        mask = base_mask.copy()
        if rs_err is not None:
            mask = mask & np.isfinite(rs_err)
            if RADIUS_SERSIC_REL_ERR_THRESH is not None:
                rel_err = np.where(rs > 0, rs_err / rs, np.inf)
                mask = mask & (rel_err <= RADIUS_SERSIC_REL_ERR_THRESH)

        all_radii.append(rs_pix[mask])

radii_unfiltered = np.concatenate(all_radii_unfiltered) if all_radii_unfiltered else np.array([])
radii_filtered = np.concatenate(all_radii) if all_radii else np.array([])

print(f"\nTotal samples (unfiltered, finite & >0): {len(radii_unfiltered)}")
print(f"Total samples (after error filtering, rel_err < {RADIUS_SERSIC_REL_ERR_THRESH}): {len(radii_filtered)}")
print(f"Filtered out: {len(radii_unfiltered) - len(radii_filtered)} samples ({100*(len(radii_unfiltered)-len(radii_filtered))/len(radii_unfiltered):.1f}%)")

if len(radii_filtered) > 0:
    print(f"\nFiltered radius_sersic (pixels):")
    print(f"  Range: [{radii_filtered.min():.4f}, {radii_filtered.max():.4f}]")
    print(f"  Mean: {radii_filtered.mean():.4f}, Std: {radii_filtered.std():.4f}")
    print(f"  Median: {np.median(radii_filtered):.4f}")
    print(f"\n  log10(radius_sersic_pix):")
    log_r = np.log10(radii_filtered)
    print(f"  Range: [{log_r.min():.4f}, {log_r.max():.4f}]")
    print(f"  Mean: {log_r.mean():.4f}, Std: {log_r.std():.4f}")

In [ ]:
# Plot radius_sersic distribution (before/after filtering)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Left: Histogram in pixel space
ax = axes[0]
ax.hist(radii_unfiltered, bins=100, alpha=0.5, label=f'Unfiltered (n={len(radii_unfiltered)})', color='gray', range=(0, 50))
ax.hist(radii_filtered, bins=100, alpha=0.7, label=f'Filtered (n={len(radii_filtered)})', color='steelblue', range=(0, 50))
ax.axvline(np.median(radii_filtered), color='red', linestyle='--', label=f'Median: {np.median(radii_filtered):.2f} px')
ax.set_xlabel('Effective Radius [pixels] (30 mas/pixel)')
ax.set_ylabel('Count')
ax.set_title('Effective Radius Distribution')
ax.legend()
ax.grid(True, alpha=0.3)

# Middle: Log-space histogram
ax = axes[1]
log_r_filt = np.log10(radii_filtered)
ax.hist(log_r_filt, bins=80, edgecolor='black', alpha=0.7, color='steelblue')
ax.axvline(log_r_filt.mean(), color='red', linestyle='--', label=f'Mean: {log_r_filt.mean():.3f}')
ax.axvline(np.median(log_r_filt), color='orange', linestyle='--', label=f'Median: {np.median(log_r_filt):.3f}')
ax.set_xlabel('log₁₀(Effective Radius [pixels])')
ax.set_ylabel('Count')
ax.set_title('log₁₀(r_eff) Distribution (filtered)')
ax.legend()
ax.grid(True, alpha=0.3)

# Right: CDF
ax = axes[2]
sorted_r = np.sort(log_r_filt)
cdf = np.arange(1, len(sorted_r) + 1) / len(sorted_r)
ax.plot(sorted_r, cdf, linewidth=2)
ax.axhline(0.5, color='gray', linestyle=':', alpha=0.5)
ax.axvline(np.median(log_r_filt), color='orange', linestyle='--', alpha=0.5)
ax.set_xlabel('log₁₀(Effective Radius [pixels])')
ax.set_ylabel('CDF')
ax.set_title('Cumulative Distribution')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 2. Dataset Class for Effective Radius

In [ ]:
class JWSTRadiusSersicDataset(Dataset):
    """JWST dataset for effective radius (radius_sersic) regression.

    Targets are log10(radius_sersic in pixels) to handle the large dynamic range.
    """
    def __init__(self, root: str, crop_size: int = 64, max_samples: int = -1, seed: int = 42,
                 rel_err_thresh: float = 0.3, effective_radius_min=2.5):
        self.crop_size = crop_size
        self.to_rgb = ToRGB()
        self.center_crop = transforms.CenterCrop(crop_size)
        self.rng = np.random.default_rng(seed=seed)
        self.rel_err_thresh = rel_err_thresh
        self.effective_radius_min = effective_radius_min

        # Load h5 files with radius_sersic
        self._files = []
        h5_files = sorted(glob.glob(os.path.join(root, "*.h5")))

        for fpath in h5_files:
            try:
                f = h5py.File(fpath, 'r')
                if 'radius_sersic' in f:
                    self._files.append(f)
            except Exception as e:
                print(f"Error: {e}")

        print(f"Loaded {len(self._files)} files with radius_sersic")

        # Build index of valid samples
        self._valid_indices = []  # (file_idx, local_idx, log10_radius_pix)

        for file_idx, f in enumerate(tqdm(self._files, desc="Indexing")):
            rs = f['radius_sersic'][:]

            # Load error
            rs_err = None
            if 'radius_sersic_err' in f:
                rs_err = f['radius_sersic_err'][:]

            # Build validity mask
            valid_mask = np.isfinite(rs) & (rs > 0)
            if rs_err is not None:
                valid_mask = valid_mask & np.isfinite(rs_err)
                if self.rel_err_thresh is not None:
                    rel_err = np.where(rs > 0, rs_err / rs, np.inf)
                    valid_mask = valid_mask & (rel_err <= self.rel_err_thresh)

            if self.effective_radius_min is not None:
                if 'radius_sersic' not in f:
                    valid_mask[:] = False
                else:
                    re = f['radius_sersic'][:]
                    re_pix = re * DEG_TO_PIXEL
                    valid_mask = valid_mask & np.isfinite(re_pix) & (re_pix >= self.effective_radius_min)
            
            # Get indices of valid samples
            valid_local_indices = np.where(valid_mask)[0]
            for local_idx in valid_local_indices:
                radius_pix = float(rs[local_idx]) * DEG_TO_PIXEL
                log_r = np.log10(radius_pix)
                if log_r >= 0 and log_r<2:
                    self._valid_indices.append((file_idx, local_idx, log_r))

        print(f"Total valid samples: {len(self._valid_indices)}")

        # Subsample if needed
        if max_samples > 0 and max_samples < len(self._valid_indices):
            indices = self.rng.choice(len(self._valid_indices), size=max_samples, replace=False)
            self._valid_indices = [self._valid_indices[i] for i in indices]
            print(f"Subsampled to {len(self._valid_indices)} samples")

    def __len__(self):
        return len(self._valid_indices)

    def __getitem__(self, index):
        file_idx, local_idx, log_radius = self._valid_indices[index]

        img = self._files[file_idx]['image'][local_idx].astype('float32')
        img = np.repeat(img[np.newaxis, :, :], 3, axis=0)
        tensor = torch.from_numpy(img)
        tensor = self.center_crop(tensor)
        tensor = torch.from_numpy(self.to_rgb(tensor.numpy()))

        return tensor, torch.tensor(log_radius, dtype=torch.float32)

## 3. Load Model and Compute Embeddings

In [ ]:
# Load model
print("Loading model...")
cfg = OmegaConf.load(MODEL_CONFIG)
model = build_model_for_eval(cfg, pretrained_weights=MODEL_WEIGHTS)
model = model.to(DEVICE)
model.eval()
print(f"Model loaded, crop_size={cfg.crops.global_crops_size}")

In [ ]:
# Create dataset
dataset = JWSTRadiusSersicDataset(
    DATA_ROOT,
    crop_size=cfg.crops.global_crops_size,
    max_samples=-1,
    seed=SEED,
    rel_err_thresh=RADIUS_SERSIC_REL_ERR_THRESH,
    effective_radius_min=3
)
print(f"Dataset size: {len(dataset)}")

In [ ]:
# Plot target distribution from dataset
log_radii_dataset = np.array([pair[2] for pair in dataset._valid_indices])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram of log10(radius_sersic_pix)
ax = axes[0]
ax.hist(log_radii_dataset, bins=50, edgecolor='black', alpha=0.7, color='steelblue')
ax.axvline(log_radii_dataset.mean(), color='red', linestyle='--', label=f'Mean: {log_radii_dataset.mean():.3f}')
ax.axvline(np.median(log_radii_dataset), color='orange', linestyle='--', label=f'Median: {np.median(log_radii_dataset):.3f}')
ax.set_xlabel('log₁₀(r_eff [pixels])')
ax.set_ylabel('Count')
ax.set_title('Target Distribution: log₁₀(Effective Radius)')
ax.legend()
ax.grid(True, alpha=0.3)

# CDF
ax = axes[1]
sorted_lr = np.sort(log_radii_dataset)
cdf = np.arange(1, len(sorted_lr) + 1) / len(sorted_lr)
ax.plot(sorted_lr, cdf, linewidth=2)
ax.axhline(0.5, color='gray', linestyle=':', alpha=0.5)
ax.axvline(np.median(log_radii_dataset), color='orange', linestyle='--', alpha=0.5)
ax.set_xlabel('log₁₀(r_eff [pixels])')
ax.set_ylabel('CDF')
ax.set_title('Cumulative Distribution')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nTarget Statistics (log₁₀(r_eff_pix)):")
print(f"  Count: {len(log_radii_dataset)}")
print(f"  Mean: {log_radii_dataset.mean():.4f}")
print(f"  Median: {np.median(log_radii_dataset):.4f}")
print(f"  Std: {log_radii_dataset.std():.4f}")
print(f"  → Corresponds to r_eff: [{10**log_radii_dataset.min():.2f}, {10**log_radii_dataset.max():.2f}] pixels")

In [ ]:
# Compute embeddings
print("Computing embeddings...")
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=8, pin_memory=True)

all_embeddings = []
all_targets = []

with torch.no_grad():
    for batch_imgs, batch_targets in tqdm(dataloader, desc="Embedding batches"):
        batch_imgs = batch_imgs.to(DEVICE)
        emb = model(batch_imgs)
        if isinstance(emb, tuple):
            emb = emb[0]
        if emb.dim() > 2:
            emb = emb.view(emb.size(0), -1)
        all_embeddings.append(emb.cpu().numpy())
        all_targets.append(batch_targets.numpy())

embeddings = np.concatenate(all_embeddings, axis=0)
targets = np.concatenate(all_targets, axis=0)

print(f"Embeddings shape: {embeddings.shape}")
print(f"Targets shape: {targets.shape}")
print(f"Target range: [{targets.min():.4f}, {targets.max():.4f}] (log₁₀(r_eff_pix))")

## 4. Linear Probe Regression

In [ ]:
# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    embeddings, targets, test_size=0.2, random_state=SEED
)
print(f"Train: {len(X_train)}, Test: {len(X_test)}")

# Convert to tensors
X_train_t = torch.tensor(X_train, dtype=torch.float32)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
y_test_t = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=128, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test_t, y_test_t), batch_size=128, shuffle=False)

# Linear regressor
class LinearRegressor(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        self.fc = nn.Linear(in_dim, 1)
    def forward(self, x):
        return self.fc(x)

in_dim = X_train.shape[1]
regressor = LinearRegressor(in_dim).to(DEVICE)
optimizer = torch.optim.Adam(regressor.parameters(), lr=1e-3, weight_decay=1e-4)
loss_fn = nn.MSELoss()

print(f"Linear Regressor: input_dim={in_dim}")

# Training
num_epochs = 50
train_losses, test_losses = [], []

for epoch in range(num_epochs):
    regressor.train()
    train_loss = 0
    for x, y in train_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        pred = regressor(x)
        loss = loss_fn(pred, y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * len(x)
    train_loss /= len(X_train)
    regressor.eval()
    test_loss = 0
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            pred = regressor(x)
            test_loss += loss_fn(pred, y).item() * len(x)
    test_loss /= len(X_test)
    train_losses.append(train_loss)
    test_losses.append(test_loss)
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{num_epochs} - Train MSE: {train_loss:.6f}, Test MSE: {test_loss:.6f}")

print(f"\nFinal Test MSE: {test_losses[-1]:.6f}")
print(f"Final Test RMSE: {np.sqrt(test_losses[-1]):.6f}")

In [ ]:
# Plot training curve
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(train_losses, label='Train MSE')
ax.plot(test_losses, label='Test MSE')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
ax.set_title('Linear Probe Training')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

# Predictions vs Ground Truth
regressor.eval()
with torch.no_grad():
    y_pred = regressor(X_test_t.to(DEVICE)).cpu().numpy().flatten()

# Metrics
from sklearn.metrics import r2_score, mean_absolute_error
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(np.mean((y_test - y_pred) ** 2))

print(f"R² Score: {r2:.4f}")
print(f"MAE: {mae:.4f}")
print(f"RMSE: {rmse:.4f}")

# Scatter plot
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_test, y_pred, alpha=0.3, s=5)
ax.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', label='Perfect prediction')
ax.set_xlabel('True log₁₀(r_eff [pixels])')
ax.set_ylabel('Predicted log₁₀(r_eff [pixels])')
ax.set_title(f'Linear Probe: R²={r2:.3f}, MAE={mae:.3f}')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

# Also show in pixel space
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(10**y_test, 10**y_pred, alpha=0.3, s=5)
ax.plot([10**y_test.min(), 10**y_test.max()], [10**y_test.min(), 10**y_test.max()], 'r--', label='Perfect prediction')
ax.set_xlabel('True r_eff [pixels]')
ax.set_ylabel('Predicted r_eff [pixels]')
ax.set_title(f'Linear Probe: R²={r2:.3f}, MAE={mae:.3f}')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

In [ ]:
# MLP regressor for comparison
HIDDEN_DIMS = [256]
MLP_LR = 1e-3
MLP_WEIGHT_DECAY = 1e-5
MLP_BATCH_SIZE = 128
MLP_EPOCHS = 60
class MLPRegressor(nn.Module):
    def __init__(self, in_dim, hidden_dims=HIDDEN_DIMS) -> None:
        super().__init__()
        layers = []
        prev_dim = in_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev_dim, h))
            layers.append(nn.ReLU(inplace=True))
            prev_dim = h
        layers.append(nn.Linear(prev_dim, 1))
        self.model = nn.Sequential(*layers)
    def forward(self, x):
        return self.model(x)

mlp = MLPRegressor(in_dim).to(DEVICE)
optimizer = torch.optim.Adam(mlp.parameters(), lr=MLP_LR, weight_decay=MLP_WEIGHT_DECAY)
loss_fn = nn.MSELoss()
mlp_train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=MLP_BATCH_SIZE, shuffle=True)
mlp_test_loader = DataLoader(TensorDataset(X_test_t, y_test_t), batch_size=MLP_BATCH_SIZE, shuffle=False)
print(f"MLP regressor hidden dims: {HIDDEN_DIMS}")

mlp_train_losses = []
mlp_test_losses = []
for epoch in range(MLP_EPOCHS):
    mlp.train()
    epoch_loss = 0
    for x_batch, y_batch in mlp_train_loader:
        x_batch, y_batch = x_batch.to(DEVICE), y_batch.to(DEVICE)
        optimizer.zero_grad()
        pred = mlp(x_batch)
        loss = loss_fn(pred, y_batch)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * len(x_batch)
    epoch_loss /= len(X_train)
    mlp.eval()
    val_loss = 0
    with torch.no_grad():
        for x_batch, y_batch in mlp_test_loader:
            x_batch, y_batch = x_batch.to(DEVICE), y_batch.to(DEVICE)
            pred = mlp(x_batch)
            val_loss += loss_fn(pred, y_batch).item() * len(x_batch)
    val_loss /= len(X_test)
    mlp_train_losses.append(epoch_loss)
    mlp_test_losses.append(val_loss)
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{MLP_EPOCHS} - Train MSE: {epoch_loss:.6f}, Test MSE: {val_loss:.6f}")

mlp.eval()
with torch.no_grad():
    y_pred = mlp(X_test_t.to(DEVICE)).cpu().numpy().flatten()

from sklearn.metrics import r2_score, mean_absolute_error
r2_mlp = r2_score(y_test, y_pred)
mae_mlp = mean_absolute_error(y_test, y_pred)
rmse_mlp = np.sqrt(np.mean((y_test - y_pred) ** 2))

print(f"\nMLP Test RMSE: {rmse_mlp:.4f}")
print(f"MLP Test R²: {r2_mlp:.4f}")
print(f"MLP Test MAE: {mae_mlp:.4f}")
print("MLP training configuration:")
print(f"  epochs: {MLP_EPOCHS}")
print(f"  batch size: {MLP_BATCH_SIZE}")
print(f"  learning rate: {MLP_LR}")
print(f"  weight decay: {MLP_WEIGHT_DECAY}")

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(mlp_train_losses, label='Train MSE', color='tab:blue')
ax.plot(mlp_test_losses, label='Test MSE', color='tab:orange')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
ax.set_title('MLP Training Curve')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_test, y_pred, alpha=0.3, s=5)
ax.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', label='Perfect prediction')
ax.set_xlabel('True log₁₀(r_eff [pixels])')
ax.set_ylabel('Predicted log₁₀(r_eff [pixels])')
ax.set_title(f'MLP Predictions (R²={r2_mlp:.3f}, MAE={mae_mlp:.3f})')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Also show in pixel space
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(10**y_test, 10**y_pred, alpha=0.3, s=5)
ax.plot([10**y_test.min(), 10**y_test.max()], [10**y_test.min(), 10**y_test.max()], 'r--', label='Perfect prediction')
ax.set_xlabel('True r_eff [pixels]')
ax.set_ylabel('Predicted r_eff [pixels]')
ax.set_title(f'Linear Probe: R²={r2:.3f}, MAE={mae:.3f}')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

## 5. Embedding Visualization (PCA, t-SNE, UMAP)

In [ ]:
# Subsample for visualization
n_vis = min(20000, len(embeddings))
vis_idx = np.random.default_rng(SEED).choice(len(embeddings), size=n_vis, replace=False)
emb_vis = embeddings[vis_idx]
targets_vis = targets[vis_idx]

print(f"Visualization samples: {n_vis}")

# PCA
print("Running PCA...")
pca = PCA(n_components=50)
emb_pca = pca.fit_transform(emb_vis)
print(f"Explained variance (top 50): {pca.explained_variance_ratio_.sum():.2%}")

# t-SNE
print("Running t-SNE...")
tsne = TSNE(n_components=2, perplexity=30, random_state=SEED, max_iter=1000, verbose=1)
emb_tsne = tsne.fit_transform(emb_pca)

# UMAP
print("Running UMAP...")
reducer = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, random_state=SEED, verbose=True)
emb_umap = reducer.fit_transform(emb_pca)
print("UMAP done.")

In [ ]:
# Plot all three, color-coded by log10(r_eff)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# PCA
ax = axes[0]
sc = ax.scatter(emb_pca[:, 0], emb_pca[:, 1], c=targets_vis, cmap='viridis', s=5, alpha=0.3)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
ax.set_title('PCA')
plt.colorbar(sc, ax=ax, label='log₁₀(r_eff [pixels])')

# t-SNE
ax = axes[1]
sc = ax.scatter(emb_tsne[:, 0], emb_tsne[:, 1], c=targets_vis, cmap='viridis', s=5, alpha=0.3)
ax.set_xlabel('t-SNE 1')
ax.set_ylabel('t-SNE 2')
ax.set_title('t-SNE')
plt.colorbar(sc, ax=ax, label='log₁₀(r_eff [pixels])')

# UMAP
fig, axes = plt.subplots(1, 1, figsize=(10, 10))
ax = axes
sc = ax.scatter(emb_umap[:, 0], emb_umap[:, 1], c=targets_vis, cmap='viridis_r', s=5, alpha=0.3)
ax.set_xlabel('UMAP 1')
ax.set_ylabel('UMAP 2')
ax.set_title('UMAP')
plt.colorbar(sc, ax=ax, label='log₁₀(r_eff [pixels])')

plt.tight_layout()
plt.show()

## 6. Summary

In [ ]:
print("=" * 60)
print("EFFECTIVE RADIUS LINEAR PROBE SUMMARY")
print("=" * 60)
print(f"\nDataset: {len(dataset)} samples")
print(f"Embedding dim: {in_dim}")
print(f"\nLinear Probe Results:")
print(f"  R² Score: {r2:.4f}")
print(f"  MAE: {mae:.4f}")
print(f"  RMSE: {rmse:.4f}")
print(f"\nInterpretation:")
if r2 > 0.5:
    print("  ✅ Strong linear relationship between embeddings and log₁₀(r_eff)")
elif r2 > 0.2:
    print("  ⚠️ Moderate linear relationship")
else:
    print("  ❌ Weak linear relationship - effective radius may not be well captured")
print("=" * 60)